# BloomFace — iniciar la copia de revisión

En Windows x64 abre esta carpeta completa en VS Code y pulsa **Run All / Ejecutar todo**. Usa un kernel Python funcional; acepta instalar ipykernel si VS Code lo solicita. La primera celda prepara Python 3.12, reconocimiento, generación y modelos sin rutas personales.

La primera preparación requiere Internet y varios GB de disco. Después reutiliza los recursos. El generador usa DirectML con respaldo en CPU. Cierra las ventanas anteriores de esta copia antes de ejecutarlo de nuevo. Consulta README.md y docs/instalacion.md.


In [ ]:
from pathlib import Path
import os
import runpy
import subprocess
import time

PREPARADO = False
candidatos = [Path.cwd(), *Path.cwd().parents]
PROYECTO = next((p.resolve() for p in candidatos
                 if (p / 'preparar_notebook.py').is_file()
                 and (p / 'bloomface/__main__.py').is_file()), None)
if PROYECTO is None:
    raise FileNotFoundError('Abre en VS Code la carpeta completa de esta copia y vuelve a ejecutar el notebook.')
preparar = runpy.run_path(str(PROYECTO / 'preparar_notebook.py'))['prepare']
PYTHON_LOCAL, SALIDA, ENTORNO, estado = preparar(PROYECTO)
PREPARADO = True
print('Reconocimiento y generación preparados. Continúa para abrir BloomFace.')


## Comprobar preparación

Espera a que la primera celda termine. La primera generación incluirá la carga de los modelos.


In [ ]:
if not PREPARADO:
    raise RuntimeError('Ejecuta primero la preparación.')
print('Reconocimiento y generación preparados. La siguiente celda abre BloomFace.')


## Abrir BloomFace

Pulsa **INICIAR → flecha → Iniciar**. Mantén uno de los cuatro gestos durante tres segundos. La ronda conserva ese gesto mientras genera. Al aparecer la flor se pausa; pulsa **Iniciar** para otra ronda. La pausa manual permite **Reanudar**.

El borde de la cámara indica el progreso y la primera flor muestra una animación de espera. La generación puede tardar más en CPU.


In [ ]:
if not PREPARADO:
    raise RuntimeError('Completa primero la preparación de ambos entornos.')
anterior = globals().get('BLOOMFACE_PROCESO')
if anterior is not None and anterior.poll() is None:
    print('BloomFace ya está abierto; busca su ventana en la barra de tareas.')
else:
    registro_app = SALIDA / 'aplicacion_mediapipe.log'
    with registro_app.open('w', encoding='utf-8') as registro:
        BLOOMFACE_PROCESO = subprocess.Popen([str(PYTHON_LOCAL), '-m', 'bloomface'],
            cwd=PROYECTO, env=ENTORNO, stdin=subprocess.DEVNULL, stdout=registro,
            stderr=subprocess.STDOUT, creationflags=subprocess.CREATE_NEW_PROCESS_GROUP)
    time.sleep(2)
    if BLOOMFACE_PROCESO.poll() is not None:
        raise RuntimeError(registro_app.read_text(encoding='utf-8', errors='replace')[-5000:])
    print('BloomFace iniciado. Pulsa INICIAR, la flecha y luego Iniciar.')


## Verificación manual

Prueba los cuatro gestos, pérdida de rostro, interrupción antes de tres segundos, pausa/reanudación y una segunda ronda. Revisa docs/verificacion.md. Estas comprobaciones requieren una persona frente a la webcam; las capturas documentales se generaron sin webcam.
